#### **TextCNN**
- CNN(합성곱 신경망): 이미지 분석시 사용하는 신경망 모델
- 이미지에서 사용되는 CNN을 이용하여 텍스트를 이미지와 같다고 가정하고 설계한 모델

---

1. 문장의 행렬화(임베딩)
    - 배치로 문장을 모은다. (배치 사이즈, 문장의 길이(토큰 개수))
    - 임베딩 처리를 통해서 (배치 사이즈, 문장의 길이, 임베딩 차원의 수)

2. 텍스트 돋보기로 문맥 읽기
    - 단어의 흐름 방향으로 구간을 선택하여 데이터 학습

3. Max Pooling (가장 큰 값을 찾는 과정)
    - 2번 과정에서의 구간 데이터들을 합성곱을 통해 가장 큰 값을 선택하는 과정
    - 가장 강한 인상 남기기

4. 최종 분류
    - 해당 데이터셋을 이용하여 최종 분류하는 과정
    - kim CNN 구조는 단어를 3, 4, 5로 분류하여 max pooling을 한 뒤 분류
    - 과적합의 위험성 때문에 dropout()을 이용하여 일정 피쳐를 0으로 만들어서 과적합 방지

##### TextCNN 사용 전 데이터 준비

1. 데이터 로드
2. 데이터 튜닝
3. 데이터 토큰화
4. 단어 사전 등록
5. 단어 사전을 이용한 인코딩

In [80]:
import pandas as pd
import re

from konlpy.tag import Komoran
from collections import Counter

1. 데이터 로드

In [81]:
df = pd.read_csv('../data/ratings_train.txt', sep = '\t')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        150000 non-null  int64
 1   document  149995 non-null  str  
 2   label     150000 non-null  int64
dtypes: int64(2), str(1)
memory usage: 3.4 MB


2. 데이터 튜닝

In [82]:
df.dropna(inplace=True)

In [83]:
def normalize(text):
    # 특수 문자, 공백에 대한 처리
    text = re.sub(r'[^가-힣0-9a-zA-Z\s\.]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [84]:
df['document'] = df['document'].map(normalize)

In [85]:
df.info()

<class 'pandas.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype
---  ------    --------------   -----
 0   id        149995 non-null  int64
 1   document  149995 non-null  str  
 2   label     149995 non-null  int64
dtypes: int64(2), str(1)
memory usage: 4.6 MB


In [86]:
df = df.loc[
    ~(df['document'] == ''),
]

In [87]:
df.drop_duplicates('document', inplace = True)

In [88]:
komoran = Komoran()

allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'MAG', 'XG']
stop_word = ['하다', '되다', '이다', '것', '수', '거']

def tokenize(text):
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos and word not in stop_word:
            tokens.append(word)
    return tokens

In [89]:
df2 = df[:1000]

In [90]:
texts, labels = df2['document'].values, df2['label'].values

# texts 토큰화
tokens_list = [tokenize(text) for text in texts]
tokens_list

[['더빙', '진짜', '짜증', '나', '목소리'],
 ['포스터', '초딩', '영화', '오버', '연기', '가볍'],
 [],
 ['교도소', '이야기', '솔직히', '재미', '없', '평점', '조정'],
 ['익살', '연기', '돋보이', '영화', '스파이더맨', '늙', '보이', '하', '커스틴 던스트', '너무나'],
 ['막', '걸음마', '떼', '초등학교', '학년', '용', '영화', '별', '반개', '아깝'],
 ['원작', '긴장감', '제대로', '살리'],
 ['반개',
  '아깝',
  '욕',
  '나오',
  '이응경',
  '길용우',
  '연기',
  '생활',
  '이',
  '정말',
  '발로',
  '납치',
  '감금',
  '반복',
  '반복',
  '이',
  '드라마',
  '가족',
  '없',
  '연기',
  '못하',
  '사람',
  '모이'],
 ['액션', '없', '재미', '있', '안', '영화'],
 ['왜', '평점', '낮', '꽤', '보', '헐리우드', '너무', '길들이', '있'],
 [],
 ['볼', '때', '눈물', '나서', '죽', '향수', '자극', '허진호', '감성', '절제', '멜로', '달인'],
 ['울', '손들', '횡단보도', '건너', '때', '뛰쳐나오', '이범수', '연기', '드럽'],
 ['좋', '신문', '기사', '로만', '보다', '보', '자꾸', '잊어버리', '사람'],
 ['취향',
  '존중',
  '진짜',
  '극장',
  '보',
  '영화',
  '가장',
  '노',
  '재',
  '노',
  '감동',
  '스토리',
  '어거지',
  '감동',
  '어거지'],
 ['매번', '긴장'],
 ['참',
  '사람',
  '웃기',
  '바스코',
  '이기',
  '락스',
  '코',
  '까',
  '고',
  '바비',
  '이기',
  '아이돌',
  '깔',
  '그냥',

In [91]:
# 최소 등장 횟수에 따른 단어 사전을 생성
freq = Counter(word for toks in tokens_list for word in toks)

In [92]:
# 최소 등장 횟수는 2회
min_count = 2

# 단어 사전에 특수 토큰 <PAD> <UNK> 토큰 대입
vocab = ['<PAD>', '<UNK>']

for word, cnt in freq.items():
    if cnt >= min_count:
        vocab.append(word)

In [93]:
len(vocab)

1001

In [94]:
stoi = {word: idx for idx, word in enumerate(vocab)}
stoi

{'<PAD>': 0,
 '<UNK>': 1,
 '더빙': 2,
 '진짜': 3,
 '짜증': 4,
 '나': 5,
 '목소리': 6,
 '포스터': 7,
 '초딩': 8,
 '영화': 9,
 '오버': 10,
 '연기': 11,
 '가볍': 12,
 '이야기': 13,
 '솔직히': 14,
 '재미': 15,
 '없': 16,
 '평점': 17,
 '돋보이': 18,
 '늙': 19,
 '보이': 20,
 '하': 21,
 '너무나': 22,
 '막': 23,
 '떼': 24,
 '초등학교': 25,
 '학년': 26,
 '용': 27,
 '별': 28,
 '반개': 29,
 '아깝': 30,
 '원작': 31,
 '긴장감': 32,
 '제대로': 33,
 '살리': 34,
 '욕': 35,
 '나오': 36,
 '생활': 37,
 '이': 38,
 '정말': 39,
 '발로': 40,
 '반복': 41,
 '드라마': 42,
 '가족': 43,
 '못하': 44,
 '사람': 45,
 '액션': 46,
 '있': 47,
 '안': 48,
 '왜': 49,
 '낮': 50,
 '꽤': 51,
 '보': 52,
 '헐리우드': 53,
 '너무': 54,
 '볼': 55,
 '때': 56,
 '눈물': 57,
 '나서': 58,
 '죽': 59,
 '향수': 60,
 '자극': 61,
 '감성': 62,
 '절제': 63,
 '멜로': 64,
 '울': 65,
 '드럽': 66,
 '좋': 67,
 '기사': 68,
 '보다': 69,
 '자꾸': 70,
 '취향': 71,
 '극장': 72,
 '가장': 73,
 '노': 74,
 '재': 75,
 '감동': 76,
 '스토리': 77,
 '어거지': 78,
 '참': 79,
 '웃기': 80,
 '이기': 81,
 '코': 82,
 '까': 83,
 '고': 84,
 '깔': 85,
 '그냥': 86,
 '난': 87,
 '이해': 88,
 '뒤': 89,
 '갈수록': 90,
 '재미없': 91,
 '이건'

In [95]:
stoi2 = dict()

for idx, word in enumerate(vocab):
    stoi2[word] = idx

stoi2

{'<PAD>': 0,
 '<UNK>': 1,
 '더빙': 2,
 '진짜': 3,
 '짜증': 4,
 '나': 5,
 '목소리': 6,
 '포스터': 7,
 '초딩': 8,
 '영화': 9,
 '오버': 10,
 '연기': 11,
 '가볍': 12,
 '이야기': 13,
 '솔직히': 14,
 '재미': 15,
 '없': 16,
 '평점': 17,
 '돋보이': 18,
 '늙': 19,
 '보이': 20,
 '하': 21,
 '너무나': 22,
 '막': 23,
 '떼': 24,
 '초등학교': 25,
 '학년': 26,
 '용': 27,
 '별': 28,
 '반개': 29,
 '아깝': 30,
 '원작': 31,
 '긴장감': 32,
 '제대로': 33,
 '살리': 34,
 '욕': 35,
 '나오': 36,
 '생활': 37,
 '이': 38,
 '정말': 39,
 '발로': 40,
 '반복': 41,
 '드라마': 42,
 '가족': 43,
 '못하': 44,
 '사람': 45,
 '액션': 46,
 '있': 47,
 '안': 48,
 '왜': 49,
 '낮': 50,
 '꽤': 51,
 '보': 52,
 '헐리우드': 53,
 '너무': 54,
 '볼': 55,
 '때': 56,
 '눈물': 57,
 '나서': 58,
 '죽': 59,
 '향수': 60,
 '자극': 61,
 '감성': 62,
 '절제': 63,
 '멜로': 64,
 '울': 65,
 '드럽': 66,
 '좋': 67,
 '기사': 68,
 '보다': 69,
 '자꾸': 70,
 '취향': 71,
 '극장': 72,
 '가장': 73,
 '노': 74,
 '재': 75,
 '감동': 76,
 '스토리': 77,
 '어거지': 78,
 '참': 79,
 '웃기': 80,
 '이기': 81,
 '코': 82,
 '까': 83,
 '고': 84,
 '깔': 85,
 '그냥': 86,
 '난': 87,
 '이해': 88,
 '뒤': 89,
 '갈수록': 90,
 '재미없': 91,
 '이건'

In [96]:
import torch

In [97]:
def encode(toks):
    result = [stoi.get(word, stoi['<UNK>']) for word in toks]
    return torch.tensor(result, dtype = torch.long)

In [98]:
enc_inputs = [ encode(toks) for toks in tokens_list ]

In [99]:
label_t = torch.tensor(labels, dtype = torch.long)

In [100]:
from sklearn.model_selection import train_test_split

In [101]:
X_train, X_test, y_train, y_test = train_test_split(
    enc_inputs, label_t, test_size = 0.2, random_state = 42, stratify = label_t
)

In [102]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# 패딩 토큰 → 토큰의 길이를 채워주기 위한 특수 토큰
from torch.nn.utils.rnn import pad_sequence

In [103]:
# 딥러닝에서 사용할 데이터를 파이토치에 맞게 변환해주는 Dataset

class TextDataset(Dataset):
    def __init__(self, xs, ys):
        # 독립 변수
        self.xs = xs
        # 종속 변수
        self.ys = ys
    
    def __len__(self):
        return len(self.xs)
    
    def __getitem__(self, idx):
        # 특정 위치에 있는 데이터를 되돌려준다. → 독립 변수, 종속 변수
        return self.xs[idx], self.ys[idx]

In [104]:
# DataLoader를 이용하여 배치 사이즈를 생성할 때 후처리 과정
MAX_K = 5

def collate_fn(batch):
    # batch의 형태는 [ (xs[0], ys[0]), (xs[1], ys[1]), ... ]
    # 독립 변수와 종속 변수로 데이터를 나눠서 저장
    xs, ys = zip(*batch)

    # 텍스트마다 토큰의 길이가 다르므로, 이를 하나의 직사각형 행렬로 맞추는 패딩 수행
    # 패딩 토큰의 위치를 변수에 저장 (일반적으로 0)
    pad_id = stoi['<PAD>']

    # 1차 토큰 채우기
    # TextCNN 모델에서 정해진 크기의 돋보기(3개 단어, 4개 단어, 5개 단어)로 구간 생성
    # 토큰의 길이가 돋보기의 크기보다 작은 경우 에러 발생 가능성
    # 이를 방지하기 위해서 모든 문장의 토큰 길이를 5로 지정(부족한 부분은 PAD 토큰으로 채워준다.)
    fixed = []
    for x in xs:
        # x: 토큰화된 문서 [token1, token2, ...]
        if len(x) < MAX_K:
            # 여기에서 필요한 패딩 토큰의 개수
            need = MAX_K - len(x)

            # 부족한 개수만큼 패딩 토큰을 x에 채워준다.
            x = torch.cat(
                [x,
                torch.full((need, ), pad_id, dtype = torch.long)]
            )
        fixed.append(x)
    
    # 2차 토큰 채우기
    # 전체적인 문서의 길이를 동일하게 맞추기 위한 패딩 토큰 작업
    # pad_sequence() 함수는 리스트 안에서 가장 긴 문장의 길이를 찾아서
    # 나머지 짧은 문장들의 빈 공간을 패딩 토큰으로 채워주는 기능
    xs_pad = pad_sequence(
        fixed,
        batch_first = True,     # 첫번째 차원을 batch size로 설정 (데이터 크기: [batch size, 최대 문장 길이])
        padding_value = pad_id
    )

    # 패딩 처리가 완료된 후 각 문장들의 실제 길이(유효 데이터 길이)를 저장(모델 연산 사용)
    lengths = torch.tensor(
        [len(x) for x in fixed], dtype = torch.long
    )

    # 결과 값은 문장 행렬, 라벨 텐서, 문장 길이
    return xs_pad, torch.stack(ys), lengths

In [105]:
train_loader = DataLoader(
    TextDataset(X_train, y_train),
    batch_size = 8,
    shuffle = True,
    collate_fn = collate_fn
)

val_loader = DataLoader(
    TextDataset(X_test, y_test),
    batch_size = 8,
    shuffle = True,
    collate_fn = collate_fn
)

딥러닝 모델 생성 (kim CNN)

In [106]:
# 학습 딥러닝 모델을 생성 (kim CNN)
    # nn.Embedding() → Conv1D(k = 3, 4, 5) → max-over-time(해당 구간에서 가장 연관이 높은 구간 선택)
        # → concat() → dropout() → Linear()

class TextCNN(nn.Module):
    def __init__(
            self, vocab_size, emb_dim, num_classes,
            kernel_size = (3, 4, 5),
            num_channel = (100),
            pad_idx = 0,
            dropout = 0.5
    ):
        # vocab_size : 단어 사전의 길이
        # emb_dim : 임베딩 벡터의 차원의 수
        # num_classes : 분류 class의 수 (이진 분류, 삼진 분류, ...)
        # kernel_size : 묶이는 단어의 개수 목록
        # num_channel : 합성곱 작업 후 출력 차원의 수
        # pad_idx : 패딩 토큰의 id 값
        # dropout : 소실되는 데이터 차원의 비율
        super().__init__()
        
        # 임베딩 가중치 행렬 생성
        # 임베딩 → 인코딩된 데이터를 무작위 vector로 만들어주는 것
        self.emb = nn.Embedding(
            vocab_size, emb_dim, padding_idx = pad_idx
        )

        # 합성곱 신경망 모델 전용 리스트를 생성
        self.convs = nn.ModuleList(
            [
                # 1차 합성곱 신경망 생성 (반복문을 이용): 텍스트를 한 방향으로 훑는 방법
                nn.Conv1d(
                    in_channels = emb_dim,       # 입력 채널의 수: 임베딩 벡터의 차원 수
                    out_channels = num_channel,  # 출력 채널의 수: 특징을 몇 명에게 물어보고 답을 받을 것인가?
                    kernel_size = k              # 단어의 구간 설정
                )
                for k in kernel_size
            ]
        )

        # 100차원 모델이 3개 생성, 출력들은 단순 열 결합의 형태
        # 300차원 → 과적합의 위험성 → 일부의 데이터를 소실(0으로 만든다)
        self.dropout = nn.Dropout(dropout)
        # 선형 모델을 이용하여 분류의 형태로 0, 1의 확률을 출력
        self.fc = nn.Linear(num_channel * len(kernel_size), num_classes)

        # 기울기 초기화
        self._init_weights()

    def _init_weights(self):
        # 임베딩 벡터 기울기 초기화 : 자비에르
        nn.init.xavier_uniform_(self.emb.weight)
        nn.init.xavier_uniform_(self.fc.weight)

        # 합성곱 모델의 기울기 초기화
        for conv in self.convs:
            # 비선형 구조에서 사용하는 다중 퍼셉트론 ReLU()를 이용하는 경우
            # 학습이 안정되도록 사용하는 초기화 방법
            nn.init.kaiming_uniform_(conv.weight)
    
    # 순전파 함수 생성
    def forward(self, x):
        # x: DataLoader를 통해서 들어오는 데이터셋 → 문장 행렬, 라벨 텐서, 문장 길이
        x = self.emb(x)
        # x는 배치 크기, 시퀀스의 길이, 임베딩 차원의 수
            # → Conv1d()에 데이터를 입력하기 위해서는 배치 크기, 임베딩 차원의 수, 시퀀스의 길이
        x = x.transpose(1, 2)

        feat_map = []
        # Conv1d 모델 → 비선형 함수 → Max 최댓값 생성 → feat_map 추가
        for conv in self.convs:
            # conv: Conv1d 모델
            h = torch.relu(conv(x))     # 배치의 크기, relu 차원의 개수, T`(시퀀스 길이 - 커널 사이즈 + 1)
            # h에서 T`의 최댓값 탐색
            h = torch.max(h, dim = 2).values    # 배치의 크기, relu 차원의 개수
            feat_map.append(h)
        
        # 열을 기준으로 feat_map 단순 결합
        z = torch.cat(feat_map, dim = 1)    # 배치의 크기, relu 차원의 수 * len(self.convs)
        
        # 과적합 방지 위해서 일부의 데이터를 소실
        z = self.dropout(z)

        # 선형 모델에 대입하여 예측
        logits = self.fc(z)

        return logits

In [107]:
# 모델 생성

model = TextCNN(
    vocab_size = len(vocab),
    emb_dim = 128,
    num_classes = 2,
    kernel_size = (3, 4, 5),
    num_channel = 64,
    pad_idx = stoi['<PAD>'],
    dropout = 0.5
)

In [108]:
# 옵티마이저 설정
optimizer = torch.optim.Adam(model.parameters(), lr = 5e-03)

# 손실 함수 설정
criterion = nn.CrossEntropyLoss()

In [109]:
# 학습, 예측을 하는 함수 선언

def run_epoch(loader, train = True):
    # loader: model에서 사용할 데이터셋
    # train: 학습 모드 / 예측 모드

    if train:
        model.train()
    else:
        model.eval()
    
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y, lengths in loader:
        # 자동 미분을 활성화할 것인가를 train 매개변수로 설정
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            # 학습 모드라면 옵티마이저, 백워드, 스텝
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += float(loss.item()) * x.size(0)

        # 예측값 → [확률, 확률] → 높은 확률의 위치
        preds = logits.argmax(dim = 1)
        correct += int( (preds == y).sum().item() )
        total = x.size(0)
    
    mean_loss = total_loss / total
    acc = correct / total
    return mean_loss, acc

In [110]:
for epoch in range(10):
    tr_loss, tr_acc = run_epoch(train_loader, True)
    val_loss, val_acc = run_epoch(val_loader, False)
    print(f"""
    Epoch: {epoch + 1}
        Train Loss: {round(tr_loss, 4)}, Train Acc: {round(tr_acc, 2)}
        Validation Loss: {round(val_loss, 4)}, Validation Acc: {round(val_acc, 2)}
""")


    Epoch: 1
        Train Loss: 64.5576, Train Acc: 60.62
        Validation Loss: 13.3175, Validation Acc: 17.88


    Epoch: 2
        Train Loss: 33.6796, Train Acc: 84.75
        Validation Loss: 15.6258, Validation Acc: 18.38


    Epoch: 3
        Train Loss: 16.6729, Train Acc: 91.88
        Validation Loss: 20.4799, Validation Acc: 17.75


    Epoch: 4
        Train Loss: 11.2893, Train Acc: 95.12
        Validation Loss: 24.047, Validation Acc: 19.12


    Epoch: 5
        Train Loss: 7.6457, Train Acc: 95.88
        Validation Loss: 28.3603, Validation Acc: 18.38


    Epoch: 6
        Train Loss: 6.1185, Train Acc: 96.62
        Validation Loss: 36.3203, Validation Acc: 19.0


    Epoch: 7
        Train Loss: 7.7704, Train Acc: 96.25
        Validation Loss: 33.9778, Validation Acc: 19.38


    Epoch: 8
        Train Loss: 20.1242, Train Acc: 94.88
        Validation Loss: 62.7538, Validation Acc: 17.5


    Epoch: 9
        Train Loss: 23.3487, Train Acc: 94.0
        Val

In [116]:
# 예측 함수 생성

@torch.no_grad()
def predict(text):
    # text : 감성 평가용 문장
    # 문장 토큰화
    toks = tokenize(text)
    # 인코딩
    ids = encode(toks)
    pad_id = stoi['<PAD>']

    # 학습된 모델에서 최대 커널의 크기 확인
    max_k = max( [ m.kernel_size[0] for m in model.convs ] )

    if len(ids) < max_k:
        need = max_k - len(ids)
        ids = torch.cat(
            [
                ids,
                torch.full(
                    (need, ), pad_id, dtype = torch.long
                )
            ]
        )
    
    # 모델에 입력으로 데이터를 대입하기 위해 차원 구조를 변경
    x = ids.unsqueeze(0)
    logits = model(x)
    prob = torch.softmax(logits, dim = 1).squeeze(0).tolist()
    pred = int(torch.argmax(logits, dim = 1).item())
    return prob, pred

In [119]:
prob, pred = predict('직원의 태도가 별로였고 실망했다')

In [120]:
print('예측값:', pred)
print('예측 확률:', prob)

예측값: 0
예측 확률: [1.0, 1.112395460441638e-19]


In [ ]:
# df 하위 100개의 데이터를 이용하여 예측값을 확인하고 정확도 계산
# 또또 처참한 내 풀이

X = df['document'].tail(100).values
y = df['label'].tail(100).values

In [125]:
prob = []
pred = []

for data in X:
    prob_, pred_ = predict(data)
    prob.append(prob_)
    pred.append(pred_)

correct = 0
for i in range(len(pred)):
    if pred[i] == y[i]:
        correct += 1

print(correct/100)

0.69


In [131]:
# 강사님 풀이
test_document = df.tail(100)['document'].values
test_label = df.tail(100)['label'].values

acc = 0

for doc, label in zip(test_document, test_label):
    _, pred = predict(doc)
    if pred == label:
        acc += 1

acc

69

In [136]:
preds = [
    predict(doc)[1] for doc in test_document
]

In [137]:
(test_label == preds).sum()

np.int64(69)